In [3]:
import ee
import os
import json
import pandas as pd
import geopandas as gpd
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np


In [5]:
PROJECT_ROOT = Path().resolve().parent

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

True

In [3]:
ee.Authenticate()



Successfully saved authorization token.


In [96]:
ee.Initialize(project=os.environ["EE_PROJECT"])

In [151]:
import importlib
import data_utils
import gee_utils

importlib.reload(data_utils)
importlib.reload(gee_utils)

<module 'gee_utils' from '/Users/bensutton/Projects/dissertation-code/src/gee_utils.py'>

In [4]:

sys.path.append(str(PROJECT_ROOT / "src"))

from gee_utils import create_images_for_all_locations, get_samples, export_patches, export_awei_p95_all_locations
from data_utils import make_padded_bbox_all_location, clean_labelled_data


NameError: name 'PROJECT_ROOT' is not defined

In [11]:
# Want to upload the sites.json to ee and then create image collection for each, clip to the bbox, need to select the dates for each location
# Could save the location in the json or just use the date from the points? 

EE_PROJECT = os.environ["EE_PROJECT"]
site_fp = PROJECT_ROOT / "configs" / "sites.json"
label_fp = PROJECT_ROOT / "configs" / "labels.gpkg"
cleaned_label_fp = PROJECT_ROOT / "configs" / "cleaned_labels.gpkg"

In [36]:
clean_labelled_data(label_fp= label_fp, cleaned_label_fp = cleaned_label_fp)

Saved a cleaned version of /Users/bensutton/Projects/dissertation-code/configs/labels.gpkg as /Users/bensutton/Projects/dissertation-code/configs/cleaned_labels.gpkg


In [12]:
# test the created cleaned labels geopackage


labels_gdf = gpd.read_file(cleaned_label_fp)

labels_gdf.head()



,label_id,longitude,latitude,location,obs_date,comparison_dates_used,s2_target_image_id,s2_old_image_id,s1_image_id,class_label,notes,created_at,updated_at,created_by,class_int,lc,geometry
0,HA0001_20210121,27.823149,-25.752589,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:04.027316+00:00,2026-05-14T14:19:04.027316+00:00,bensutton,3,1,POINT (27.82315 -25.75259)
1,HA0002_20210121,27.807339,-25.760474,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:06.695977+00:00,2026-05-14T14:19:06.695977+00:00,bensutton,3,1,POINT (27.80734 -25.76047)
2,HA0003_20210121,27.809334,-25.755411,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:10.017332+00:00,2026-05-14T14:19:10.017332+00:00,bensutton,3,1,POINT (27.80933 -25.75541)
3,HA0004_20210121,27.805043,-25.758155,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:13.638342+00:00,2026-05-14T14:19:13.638342+00:00,bensutton,3,1,POINT (27.80504 -25.75816)
4,HA0005_20210121,27.819760,-25.762774,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:18.937682+00:00,2026-05-14T14:19:18.937682+00:00,bensutton,3,1,POINT (27.81976 -25.76277)


In [14]:
print(len(labels_gdf))
labels_gdf.groupby(["location", "class_label"])["lc"].count().reset_index(name="count")

16192


,location,class_label,count
0,Hartbeespoort,LEV,509
1,Hartbeespoort,floating_plants,1002
2,Hartbeespoort,open_water,440
3,Hartbeespoort,surface_algae,81
4,Inle,LEV,500
5,Inle,floating_plants,1000
6,Inle,open_water,500
7,Mula,LEV,576
8,Mula,floating_plants,1003
9,Mula,open_water,493


In [16]:

labels_gdf.groupby(["location"])["lc"].count().reset_index(name="count")

,location,count
0,Hartbeespoort,2032
1,Inle,2000
2,Mula,2072
3,RawaPening,2004
4,Rodman,2042
5,Valsequillo,2040
6,Vembanad,2002
7,Winam,2000


In [40]:
# Make a bounding box for each site based on the total bounds of the labelled points and add a padding

make_padded_bbox_all_location(sites_file= site_fp, project_root= PROJECT_ROOT)

Created padded bbox from the points files for Vembanad, Winam, Inle, Hartbeespoort, Mula, RawaPening, Rodman, Valsequillo


In [ ]:
# Memomry limits made it neccesary to create an AWEI_p95 (with the 95th percentile for AWEIsh, 2019-2025) image that is saved to assets for each location.
# After running the code to export use the returned task list areto ensure exports are completed before running create_images_for_all_locations()

task_list = export_awei_p95_all_locations(ee_project= EE_PROJECT, sites_file= site_fp)

In [ ]:
for task in task_list:
    print(task.status())

In [ ]:
image_collection = create_images_for_all_locations(sites_file= site_fp, cleaned_label_fp=cleaned_label_fp, ee_project=EE_PROJECT, clip = False)

In [147]:
print(image_collection.first().bandNames().getInfo())

['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'AWEIp95']


In [ ]:
# Due to memory limits it is not possible to convert all the sampled points to a data frame, therefore the sampled points for each location 
# is export to drive as a CSV separately


# Trying to save all without aweip95

get_samples(merged_ic=image_collection,cleaned_label_fp= cleaned_label_fp)

In [ ]:
for task in ee.batch.Task.list():
    status = task.status()
    desc = status.get("description", "")

    if desc.startswith("sampled_points_"):
        print(desc, status["state"])
        print("EECU seconds:", status.get("batch_eecu_usage_seconds"))
        if status["state"] == "FAILED":
            print(status.get("error_message"))

In [18]:
# Load location samples from csvs in outputs/sampled_points_by_location, combine into a single dataframe and save 

samples_outpath = PROJECT_ROOT / "outputs" / "sample_points.csv"

with open(site_fp) as f:
    sites= json.load(f)

location_list = list(sites.get("sites", {}).keys())

list_of_location_dfs = []

for location in location_list:
    location_samples_fp = PROJECT_ROOT / "outputs/sampled_points_by_location/sampled_points" / f"sampled_points_{location}.csv" 

    location_df = pd.read_csv(location_samples_fp)

    list_of_location_dfs.append(location_df)

all_samples = pd.concat(list_of_location_dfs, ignore_index= True)



all_samples.to_csv(samples_outpath, index= False)



In [22]:
# check len of inle 

inle_fp = PROJECT_ROOT / "outputs/sampled_points_by_location/sampled_points" / f"sampled_points_inle.csv"

inle_points = pd.read_csv(inle_fp)
len(inle_points)

1830

In [19]:
# Number of samples per class per location
all_samples_fp = PROJECT_ROOT / "outputs" / "sample_points.csv"
all_samples = pd.read_csv(all_samples_fp)

print(len(all_samples))
all_samples.groupby(["location"])["lc"].count().reset_index(name="count")

16019


,location,count
0,Hartbeespoort,2032
1,Inle,1830
2,Mula,2072
3,RawaPening,2002
4,Rodman,2042
5,Valsequillo,2040
6,Vembanad,2002
7,Winam,1999


In [ ]:
export_patches(merged_ic = image_collection, cleaned_label_fp= cleaned_label_fp, kernel_size= 15)

In [ ]:
export_patches(merged_ic = image_collection, cleaned_label_fp= cleaned_label_fp, kernel_size= 31)

In [134]:
asset_id = 'projects/' + EE_PROJECT + '/assets/awei_p95_Winam'

AWEI_image = ee.Image(asset_id)

display('bands', AWEI_image.bandNames())

'bands'

In [ ]:
stats = AWEI_image.reduceRegion(
    reducer=ee.Reducer.minMax()
        .combine(
            reducer2=ee.Reducer.mean(),
            sharedInputs=True
        ),
    geometry=AWEI_image.geometry(),
    scale=10,      # use the appropriate resolution
    maxPixels=1e13
)

print(stats.getInfo())

{'AWEIp95_max': 12159.125, 'AWEIp95_mean': 955.1331094299671, 'AWEIp95_min': -5204}


In [ ]:
# For combining GEOJSON

feature_list = []


location_list = ["Hartbeespoort", "Rodman","Mula", "Inle", "Vembanad", "Valsequillo", "RawaPening"]

for location in location_list:
    location_fp = PROJECT_ROOT / "outputs/31px_patches" / f"sampled_31_pixel_patches_for_{location}.geojson"
    with open(location_fp, "r") as f:
        patches = json.load(f)

        for feature in patches["features"]:
            feature["id"] = feature["properties"]["label_id"]
            feature_list.append(feature)

combined_dict = {"type": "FeatureCollection", "features": feature_list}

outpath = PROJECT_ROOT / "outputs/15px_patches" / "combined_15px_patches.geojson"

with open(outpath, 'w') as f:
    json.dump(combined_dict, f)